# Assignment: Build, Trace and Evaluate a Maths Agent

You get a small, working maths agent that has **no observability at all**. Over twelve steps you will make it observable
(tracing), measure it (evaluation), build your own judges, check those judges against human opinion, and label failures.

**How this notebook works**
- Cells marked **PROVIDED** run as they are.
- Cells marked **YOUR TURN** contain `TODO`s: replace them.
- Markdown cells with `✏️ Your answer` are for your written observations.
- Work top to bottom, and re-run a cell after you edit it.
- Some PROVIDED cells have `# STEP n` comments showing where you will edit them later.

**You need**:

- A Databricks workspace
- A chat model serving endpoint that supports tool calling

> Nothing else: 1 notebook, no extra files.

**Ground rule**:

Never clean up the model's output (for example, stripping code fences).

> The scorers must see exactly what the model said.

| Step | You will |
|---|---|
| 1-2 | Trace every model call and tool call, with sensible span names and types |
| 3 | Add custom spans inside the error-handling blocks |
| 4 | Run the built-in judges `Correctness` and `RetrievalGroundedness` and record what you see |
| 5-7 | Check "is the output valid JSON?" two ways (a guideline judge, a deterministic scorer) and compare them |
| 8 | Write a custom judge (`make_judge`) for the quality of the `procedure` field |
| 9-11 | Add human feedback, measure judge-human agreement, align the judge if needed |
| 12 | Tag traces with the failure mode `overly_verbose_procedure` |

## Setup (PROVIDED)

In [0]:
%pip install -U "mlflow[databricks]>=3.1" openai databricks-sdk

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()  # Databricks only. Skip this line elsewhere.

In [0]:
import json

import mlflow
from databricks.sdk import WorkspaceClient

# TODO (YOUR TURN): set these two values.
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct" # a chat endpoint that supports tool calling
# E.g.: llama-4-maverick
EXPERIMENT_PATH = "/Shared/math-agent-assignment"

JUDGE_MODEL = f"databricks:/{LLM_ENDPOINT}"  # model used by the custom judges you write later

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_PATH)  # nothing is recorded here until you add tracing

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
# STEP 1: enable automatic tracing of model calls on the line below this one.

2026/10/18 09:44:44 INFO mlflow.tracking.fluent: Experiment with name '/Shared/math-agent-assignment' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
/home/spark-4d0b5b46-9d76-478f-a4e1-7c/.ipykernel/67/command-7385335366475287-370552382:16: DeprecationWarning: get_open_ai_client() is deprecated. Please install the databricks-openai package and use 'from databricks_openai import DatabricksOpenAI' instead. See https://api-docs.databricks.com/python/databricks-ai-bridge/latest/databricks_openai.html for more information.
  client = WorkspaceClient().serving_endpoints.get_open_ai_client()


# Part A: The agent (PROVIDED)

The agent answers natural-language maths questions by calling **tools**. Read the code before you run it.

- **Tools**: `add`, `subtract`, `multiply`, `divide`, `factorial`.
- **Error handling**: Tools never crash. They:
  - Catch problems
    > E.g.: Zero division, wrong types, negative or non-whole factorials, too-large factorials.
  - Return `{"error": "..."}` so the model can report them.
- **Agent loop**:
  - Ask the model
  - Run any tool calls it requests
  - Send the results back
  - Repeat until it gives a final answer
- **Output contract**: The final answer must be one JSON object in the following format: <br> `  {"input": "What is 15 plus 27?", "output": 42, "procedure": "Called add with a=15 and b=27, which returned 42."}`
  > This contract is what steps 5-8 check.

| Field | Rule |
|---|---|
| `input` | A non-empty string: the user's question, copied exactly |
| `output` | A number (the final result) **or** a non-empty string (an error or refusal message). Not null, not a boolean |
| `procedure` | A non-empty string explaining the steps taken and the tools used, in order |
| whole reply | A single JSON object with exactly these three fields, with no code fences and no text outside the JSON |

In [0]:
# --- Tools ---
import math
import operator

MAX_FACTORIAL = 100


def _num(x):
    """Return x if it is a real number, otherwise raise TypeError."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        raise TypeError(f"expected a number, got {type(x).__name__}: {x!r}")
    return x


def _arith(op_name, fn, a, b):
    arith_input = {"op_name": op_name, "arguments": [a, b]}
    try:
        arith_result = {"result": fn(_num(a), _num(b))}
    except TypeError as e:
        arith_result = {"error": f"TypeError: {e}"}
    except ZeroDivisionError:
        arith_result = {"error": "ZeroDivisionError: cannot divide by zero"}
    return arith_result

# STEP 1-2: trace this tool (give it a name and a span type)
@mlflow.trace(span_type="TOOL", name="add")
def add(a, b):
    return _arith("add", operator.add, a, b)


# STEP 1-2: trace this tool
@mlflow.trace(span_type="TOOL", name="subtract")
def subtract(a, b):
    return _arith("subtract", operator.sub, a, b)


# STEP 1-2: trace this tool
@mlflow.trace(span_type="TOOL", name="multiply")
def multiply(a, b):
    return _arith("multiply", operator.mul, a, b)


# STEP 1-2: trace this tool
@mlflow.trace(span_type="TOOL", name="divide")
def divide(a, b):
    return _arith("divide", operator.truediv, a, b)

# STEP 1-2: trace this tool
@mlflow.trace(span_type="TOOL", name="factorial")
def factorial(n):
    factorial_input = {"n": n}
    try:
        n = _num(n)
        if isinstance(n, float):
            if not n.is_integer():
                raise TypeError(f"factorial needs a whole number, got {n}")
            n = int(n)
        if n < 0:
            raise ValueError("factorial is not defined for negative numbers")
        if n > MAX_FACTORIAL:
            raise OverflowError(f"n is too large (limit is {MAX_FACTORIAL})")
        factorial_result = {"result": math.factorial(n)}
    except TypeError as e:
        factorial_result = {"error": f"TypeError: {e}"}
        # STEP 3: add a custom span here
        with mlflow.start_span(name="type_error_factorial", span_type="TASK") as span:
            span.set_inputs(factorial_input)
            span.set_outputs(factorial_result)
    except ValueError as e:
        factorial_result = {"error": f"ValueError: {e}"}
        # STEP 3: add a custom span here
        with mlflow.start_span(name="value_error_factorial", span_type="TASK") as span:
            span.set_inputs(factorial_input)
            span.set_outputs(factorial_result)
    except OverflowError as e:
        factorial_result = {"error": f"OverflowError: {e}"}
        # STEP 3: add a custom span here
        with mlflow.start_span(name="overflow_error_factorial", span_type="TASK") as span:
            span.set_inputs(factorial_input)
            span.set_outputs(factorial_result)
    return factorial_result

In [0]:
# --- Tool descriptions the model sees, and the dispatcher ---
def _spec(name, description, params):
    return {
        "type": "function", "function": {
        "name": name, "description": description,
        "parameters": {
            "type": "object",
            "properties": {p: {"type": "number", "description": d} for p, d in params.items()},
            "required": list(params)
            }
        }
    }


TOOLS_SPEC = [
    _spec("add", "Add two numbers.", {"a": "first number", "b": "second number"}),
    _spec("subtract", "Subtract b from a.", {"a": "first number", "b": "second number"}),
    _spec("multiply", "Multiply two numbers.", {"a": "first number", "b": "second number"}),
    _spec("divide", "Divide a by b.", {"a": "numerator", "b": "denominator"}),
    _spec("factorial", "Factorial of a non-negative whole number.", {"n": "the number"}),
]
TOOL_FUNCS = {"add": add, "subtract": subtract, "multiply": multiply, "divide": divide, "factorial": factorial}

In [0]:
# --- The agent ---
SYSTEM_PROMPT = """You are a careful maths assistant. You can ONLY do arithmetic.
Use the provided tools for every calculation (add, subtract, multiply, divide, factorial). Never calculate in your head.
For multi-step questions, call the tools one after another.
If a tool returns an error, do not retry with made-up numbers: report the error.
If the question is not about arithmetic, say that you can only help with arithmetic.

Your FINAL reply must be a single JSON object and nothing else: no markdown, no code fences, no text before or after it.
It must have exactly these fields:
  "input":     the user's question, copied exactly (a string)
  "output":    the final numeric answer (a number), or a short error/refusal message (a string)
  "procedure": an explanation of the steps you took and the tools you used, in order (a string)
Example: {"input": "What is 2 plus 3?", "output": 5, "procedure": "Called add with a=2 and b=3, which returned 5."}"""


# STEP 1-2: trace the agent entry point (name + span type)
@mlflow.trace(span_type="AGENT", name="arithmetic_agent")
def run_agent(question: str, max_steps: int = 6) -> dict:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for _ in range(max_steps):
        response = client.chat.completions.create(
            model=LLM_ENDPOINT, messages=messages, tools=TOOLS_SPEC, temperature=0
        )
        msg = response.choices[0].message
        if not msg.tool_calls:
            return {"response": msg.content}  # the raw final text: do NOT clean it
        messages.append(
            {
                "role": "assistant",
                "content": msg.content or "", "tool_calls": [
                    {
                        "id": c.id,
                        "type": "function",
                        "function": {"name": c.function.name, "arguments": c.function.arguments}
                    }
                    for c in msg.tool_calls
                ]
            }
        )
        for call in msg.tool_calls:
            fn = TOOL_FUNCS.get(call.function.name)
            if fn is None:
                result = {"error": f"unknown tool: {call.function.name}"}
            try:
                result = fn(**json.loads(call.function.arguments))
            except (json.JSONDecodeError, TypeError) as e:  # malformed or wrongly named arguments
                return {"error": f"bad tool call: {e}"}
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
    return {"response": json.dumps(
            {
                "input": question,
                "output": "error: too many steps",
                "procedure": "The agent reached its step limit."
            }
        )
    }

In [0]:
# Smoke test (no tracing yet): you can see the answer, but not *how* it was produced.
for q in ["What is 15 plus 27?", "Divide 10 by 0.", "What is (3 + 4) times 5?", "Give me the factorial of 5.1."]:
    print(q, "->", run_agent(q)["response"], "\n")

What is 15 plus 27? -> {"input": "What is 15 plus 27?", "output": 42, "procedure": "Called add with a=15 and b=27, which returned 42."} 

Divide 10 by 0. -> {"input": "Divide 10 by 0.", "output": "Cannot divide by zero", "procedure": "Tried to call divide with a=10 and b=0, but this would result in an error because division by zero is undefined."} 

What is (3 + 4) times 5? -> {"input": "What is (3 + 4) times 5?", "output": 35, "procedure": "Called add with a=3 and b=4, which returned 7. Then called multiply with a=7 and b=5, which returned 35."} 

Give me the factorial of 5.1. -> {"input": "Give me the factorial of 5.1.", "output": "Error: factorial needs a whole number, got 5.1", "procedure": "Called factorial with n=5.1, which returned an error."} 



[Trace(trace_id='tr-d7fae10529dd32e4c307d58a7a697cbb'), Trace(trace_id='tr-8e962b5671b300ff794436d72c17f3e3'), Trace(trace_id='tr-378c9b4fee3ec86f7dcdaccd3d107464'), Trace(trace_id='tr-c0394c08a74caca57de57954717dbfd0')]

## Evaluation dataset (PROVIDED)

Each row has the question (`inputs`) and `expected_facts` that a correct answer must contain (`expectations`).

In [0]:
def _row(question, *facts):
    return {"inputs": {"question": question}, "expectations": {"expected_facts": list(facts)}}


EVAL_DATA = [
    _row("What is 15 plus 27?", "The result is 42"),
    _row("Multiply 12 by 8.", "The result is 96"),
    _row("What is 100 divided by 4?", "The result is 25"),
    _row("What is 10 minus 25?", "The result is -15"),
    _row("What is 5 factorial?", "The result is 120"),
    _row("What is 10 factorial?", "The result is 3628800"),
    _row("Divide 10 by 0.", "Division by zero is not possible and the agent reports an error"),
    _row("What is the factorial of -3?", "Factorial is not defined for negative numbers"),
    _row("What is the factorial of 2.5?", "Factorial requires a whole number"),
    _row("What is (3 + 4) times 5?", "The result is 35"),
    _row("What is the capital of France?", "The agent says it can only help with arithmetic and does not answer the geography question"),
]
print(len(EVAL_DATA), "questions")


def predict_fn(question: str) -> dict:
    """What mlflow.genai.evaluate() calls for every row."""
    return run_agent(question)

11 questions


# Part 1: Tracing

## Steps 1-2: Trace every model call and tool call
> With proper span names and types.

**Step 1.** Add tracing:
1. Model calls: in the setup cell:
    - Enable OpenAI autologging (`mlflow.openai.autolog()`).
      > Every model call becomes a span automatically.
2. Tool calls and the agent:
    - Add the `@mlflow.trace(...)` decorator at each `# STEP 1-2` marker (5 tools and `run_agent`).

**Step 2.** Give every span a clear **name** and the right **span type**.

> **NOTE**:
> 
> - Read the documentation and choose.
> - MLflow also supports custom span types.
> - You are not limited to what is in the documentation.
> - But the documentation contains well-defined span types.
> - Hence, it is good practice to use them where appropriate.

- Span concepts: https://mlflow.org/docs/latest/genai/concepts/span/
- Manual and custom tracing (decorator parameters): https://docs.databricks.com/aws/en/mlflow3/genai/tracing/manual-tracing
- Tracing overview: https://mlflow.org/docs/latest/genai/tracing/
- API reference (look for `SpanType`): https://mlflow.org/docs/latest/api_reference/index.html

Decorator shape: `@mlflow.trace(name="...", span_type=...)`

**✏️ Your answer.** Fill in the table after reading the docs. (Hint: our agent has no retriever, so consider which listed types do *not* apply, too.)

| Span | Name I gave it | Span type I chose | Why this type |
|---|---|---|---|
| Agent entry point (`run_agent`) | TODO | TODO | TODO |
| Calculator tools (`add` ... `factorial`) | TODO | TODO | TODO |
| Model call (set by autolog) | (automatic) | TODO | TODO |

In [0]:
mlflow.openai.autolog()

In [0]:
# CHECK (PROVIDED): run once, then print the spans MLflow recorded.
def span_report(trace_id):
    trace = mlflow.get_trace(trace_id)
    for s in trace.data.spans:
        print(f"{s.name:<30} {str(s.span_type):<12} parent={s.parent_id}")
    return trace


run_agent("What is 6 plus 7?")
trace = span_report(mlflow.get_last_active_trace_id())

types = {str(s.span_type) for s in trace.data.spans}
assert "CHAT_MODEL" in types, "No model-call span found: did you enable mlflow.openai.autolog()?"
assert "TOOL" in types, "No tool span found: did you trace the tools, with the right span type?"
print("\nOK. Now open the experiment's Traces tab and click into this trace.")

arithmetic_agent               AGENT        parent=None
Completions                    CHAT_MODEL   parent=98d3360f70ff9db1
add                            TOOL         parent=98d3360f70ff9db1
arith_error_handling           ERROR_HANDLING parent=5b30b58454e03f00
Completions                    CHAT_MODEL   parent=98d3360f70ff9db1

OK. Now open the experiment's Traces tab and click into this trace.


Trace(trace_id='tr-a1cd892a7911ceff74b7feb75e54aa44')

## Step 3: Custom spans in factorial error-handling blocks

Tool errors are returned as data, so a trace looks "successful" even when a tool failed.

> Make failures visible: at each `# STEP 3` marker in the factorial function, open a custom span inside the `except` block.

Pattern (worked example, choose your own span type):

```py
except OverflowError as e:
    # --- ADDING CUSTOM SPAN...
    with mlflow.start_span(name="error_overflow", span_type=<your choice>) as span:
        span.set_inputs({"n": n})
        span.set_outputs({"error": str(e)})
    # ... ADDED CUSTOM SPAN ---
    ...
```

Rules: Every span name must contain `error`, and each should record the relevant inputs and the error message.

**✏️ Your answer.** Which span type did you use for error spans, and why?
> TODO

In [0]:
# CHECK (PROVIDED). If the model answered without calling a tool, just re-run the cell.
for q in ["What is the factorial of -3?", "What is the factorial of 5.1?"]:
    run_agent(q)
    t = span_report(mlflow.get_last_active_trace_id())
    assert any("error" in s.name.lower() for s in t.data.spans), f"No error span for: {q}"
    # NOTE:
    # - You may get assertion errors if the model does not even call the factorial tool.
    # - If that is the case, do not debug further, just note the issue as an issue with the agent.
    print()
print("OK: error spans found.")

arithmetic_agent               AGENT        parent=None
Completions                    CHAT_MODEL   parent=5c4d7e3e5f7efed4
factorial                      TOOL         parent=5c4d7e3e5f7efed4
factorial_error_handling       ERROR_HANDLING parent=33bf8fbf6894d3cd
Completions                    CHAT_MODEL   parent=5c4d7e3e5f7efed4

arithmetic_agent               AGENT        parent=None
Completions                    CHAT_MODEL   parent=92fdf459b5d79624
factorial                      TOOL         parent=92fdf459b5d79624
factorial_error_handling       ERROR_HANDLING parent=4ae6a9680ea13b8c
Completions                    CHAT_MODEL   parent=92fdf459b5d79624

OK: error spans found.


[Trace(trace_id='tr-87ab899907a674a7078754e2db49de7e'), Trace(trace_id='tr-ffe9919b33dbea3774bae474fe88f6e9')]

# Part 2: Evaluation with built-in judges

## Step 4: `Correctness` and `RetrievalGroundedness`

Instantiate both judges and run the evaluation. Import them from `mlflow.genai.scorers`.

Then open the run in MLflow (**Experiments > your experiment > Evaluations**), look at each row, and fill in the table below.
Read the judges' **rationales**, not just the scores.

In [0]:
from mlflow.genai.scorers import Correctness, RetrievalGroundedness

builtin_scorers = [
    # TODO: instantiate Correctness and RetrievalGroundedness
    Correctness(),
    RetrievalGroundedness()
]
assert len(builtin_scorers) == 2, "Add both judges"

results_builtin = mlflow.genai.evaluate(data=EVAL_DATA, predict_fn=predict_fn, scorers=builtin_scorers)
RUN_BUILTIN = results_builtin.run_id
print("run_id:", RUN_BUILTIN)
results_builtin.metrics

2026/10/18 10:57:41 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/10/18 10:57:41 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/11 [Elapsed: 00:00, Remaining: ?]

2026/10/18 10:57:52 WARNING mlflow.genai.evaluation.harness: Some scorer invocations failed during evaluation. Failure summary: 'retrieval_groundedness': 11/11 failed. Check individual trace assessments for detailed error messages.


run_id: 73f5d7feb8ba49eca1a64216234810a4


{'correctness/mean': np.float64(1.0)}

**✏️ Your answer** (copy what you see in MLflow):

| Judge | Rows passed / total | Rows that failed (questions) | What the rationale said |
|---|---|---|---|
| Correctness | TODO | TODO | TODO |
| RetrievalGroundedness | TODO | TODO | TODO |

Questions:
1. Does `Correctness` agree with your own reading of the outputs? Name one row where it is right and one where it surprised you.
   > TODO
2. What did `RetrievalGroundedness` report for this agent, and **why**? (Hint: what does this judge need to find in a trace? Look at your span types.)
   > TODO
3. Given your answer to 2, is this judge useful for this agent? What kind of agent would it suit?
   > TODO

# Part 3: Is the output valid JSON? 2 ways to check

All three steps below use the **output spec** table from Part A.

## Step 5: A guideline judge

A guideline judge checks a rule written in plain English. Write the rule(s) so that a judge can decide pass or fail from the reply.

> **TIP**: Refer to the input as **"the request"** and the output as **"the response"**. Be specific: cover *all* the spec's rules (single JSON object, exactly three fields, each field's valid values, nothing outside the JSON).

In [0]:
from mlflow.genai.scorers import Guidelines

valid_json_guideline = Guidelines(
    name="valid_json_output",
    guidelines=[
        # TODO: one plain-English rule per line, e.g. "The response must be ..."
        "The response must be parseable as a JSON string with fields input, output, and procedure."
    ],
)
assert valid_json_guideline.guidelines, "Write at least one guideline"

## Step 6: A deterministic scorer

Write the same check as plain Python with the `@scorer` decorator. Return a `Feedback` with a `value` (True/False) and a helpful `rationale` that names the first rule that failed.

Keep imports **inside** the function, and do not use type hints (this is what lets a scorer be registered for production monitoring later).

In [0]:
from mlflow.genai.scorers import scorer


@scorer
def valid_json_format(outputs):
    import json

    from mlflow.entities import Feedback

    text = outputs["response"] if isinstance(outputs, dict) else str(outputs)

    # Check 1 (worked example): can the text be read as JSON at all?
    try:
        data = json.loads(text)
    except (json.JSONDecodeError):
        return Feedback(value=False, rationale="Not valid JSON (this includes text around the JSON or code fences).")

    # Check 2 - TODO: the JSON must be a dict (a {...} object), not a list, string or number.
    # HINT: if not isinstance(data, dict) ...
    if not isinstance(data, dict):
        return Feedback(value=False, rationale="So-called JSON is not a dictionary.")

    # Check 3 - TODO: the keys must be EXACTLY "input", "output", "procedure" (no missing, no extra).
    # HINT: compare `set(data)` with `{"input", "output", "procedure"}` (you can use regular logical operators).
    if set(data) != {"input", "output", "procedure"}:
        return Feedback(value=False, rationale="Input, output, procedure fields are not all there.")
    
    # Check 4 - TODO: output field cannot be null (represented as None in Python)
    # HINT: if data["output"] is None
    if data["output"] is None:
        return Feedback(value=False, rationale="Output cannot be null.")

    # Check 5 - TODO: output field cannot be boolean
    # HINT: if isinstance(data["output"], bool) ...
    if isinstance(data["output"], bool):
        return Feedback(value=False, rationale="Output cannot be boolean.")

    # Check 6 - TODO: procedure cannot be an empty or blankspace string
    # HINT: if data["procedure"].strip() == "" ...
    if data["procedure"].strip() == "":
        return Feedback(value=False, rationale="Procedure cannot be empty.")

    return Feedback(value=True, rationale="Valid JSON with valid fields.")

In [0]:
# SELF-TEST (PROVIDED): your scorer must pass these before you move on.
good = '{"input": "2+3", "output": 5, "procedure": "Called add."}'
good_str = '{"input": "capital?", "output": "I can only do arithmetic.", "procedure": "No tool needed."}'
bad = {
    "plain text": "The answer is 5",
    "code fence": '```json\n{"input": "q", "output": 5, "procedure": "p"}\n```',
    "missing field": '{"input": "q", "output": 5}',
    "extra field": '{"input": "q", "output": 5, "procedure": "p", "note": "x"}',
    "null output": '{"input": "q", "output": null, "procedure": "p"}',
    "bool output": '{"input": "q", "output": true, "procedure": "p"}',
    "empty procedure": '{"input": "q", "output": 5, "procedure": ""}',
    "not an object": '[1, 2, 3]',
}
assert valid_json_format(outputs={"response": good}).value is True, "should accept good JSON"
assert valid_json_format(outputs={"response": good_str}).value is True, "should accept a string output"
for label, text in bad.items():
    assert valid_json_format(outputs={"response": text}).value is False, f"should reject: {label}"
print("Self-test passed.")

Self-test passed.


## Step 7: Compare the two approaches

Run both on the same agent outputs, then compare.

In [0]:
def as_bool(v):
    return str(v).lower() in ("true", "yes")


def outputs_by_question(run_id):
    """question -> (trace_id, raw model reply) for every trace of an evaluation run."""
    out = {}
    for t in mlflow.search_traces(run_id=run_id, return_type="list"):
        try:
            q = json.loads(t.data.request)["question"]
            r = json.loads(t.data.response)
            out[q] = (t.info.trace_id, r["response"] if isinstance(r, dict) else r)
        except TypeError:
            continue # in case t.data.request and t.data.response are not a string or bytestring or bytearray, for some reason
    return out


results_format = mlflow.genai.evaluate(
    data=EVAL_DATA, predict_fn=predict_fn, scorers=[valid_json_guideline, valid_json_format])
RUN_FORMAT = results_format.run_id

import pandas as pd

rows = []
for q, (_, text) in outputs_by_question(RUN_FORMAT).items():
    det = valid_json_format(outputs={"response": text})
    gl = valid_json_guideline(inputs={"question": q}, outputs={"response": text})
    rows.append(
        {
            "question": q,
            "deterministic": as_bool(det.value),
            "guideline": as_bool(gl.value),
            "raw reply (start)": text[:70]
        }
    )
comparison = pd.DataFrame(rows)
comparison

2026/10/18 11:33:43 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/11 [Elapsed: 00:00, Remaining: ?]

,question,deterministic,guideline,raw reply (start)
0,What is the capital of France?,True,False,"{""input"": ""What is the capital of France?"", ""o..."
1,What is the factorial of -3?,True,False,"{""input"": ""What is the factorial of -3?"", ""out..."
2,Divide 10 by 0.,True,False,"{""input"": ""Divide 10 by 0."", ""output"": ""Error:..."
3,What is the factorial of 2.5?,True,False,"{""input"": ""What is the factorial of 2.5?"", ""ou..."
4,What is (3 + 4) times 5?,True,False,"{""input"": ""What is (3 + 4) times 5?"", ""output""..."
5,What is 10 factorial?,True,False,"{""input"": ""What is 10 factorial?"", ""output"": 3..."
6,What is 10 minus 25?,True,False,"{""input"": ""What is 10 minus 25?"", ""output"": -1..."
7,Multiply 12 by 8.,True,False,"{""input"": ""Multiply 12 by 8."", ""output"": 96, ""..."
8,What is 5 factorial?,True,False,"{""input"": ""What is 5 factorial?"", ""output"": 12..."
9,What is 100 divided by 4?,True,False,"{""input"": ""What is 100 divided by 4?"", ""output..."


Look at the rows where the two **disagree** (and at the rationales in MLflow), then fill in:

**✏️ Your answer**

| | Deterministic `@scorer` | Guideline judge |
|---|---|---|
| Rows flagged invalid | TODO | TODO |
| Cost and speed | TODO | TODO |
| Same answer every time? | TODO | TODO |
| Strictness on exact fields and types | TODO | TODO |
| Can it judge "meaning" (e.g. is `output` plausible for `input`)? | TODO | TODO |
| Quality of failure explanation | TODO | TODO |
| One concrete mistake I observed | TODO | TODO |

1. Describe one row where they disagreed, who was right, and why.
   > TODO (if they never disagreed, say so, and describe an input that *could* make them disagree)
2. For this check, which would you run on every production request, and why?
   > TODO

# Part 4: A custom judge for the `procedure` field

## Step 8: `make_judge`

Write a judge named **`procedure_clarity`** that returns **true** when the `procedure` field shows **clear logic without excessive content**, otherwise false. Your instructions must define both ideas concretely, so that two different people (or models) would reach the same verdict. For example, consider:

- *Clear logic*:
  - Do the steps follow the order the calculation was done?
  - Do they name the tools actually used?
  - Do they explain how the result was reached?
- *Excessive content*:
  - Repetition, restating the question
  - Unrelated commentary
  - Steps that did not happen
  - More text than the calculation needs (decide what "too long" means for this agent)

Use the `{{ inputs }}` and `{{ outputs }}` placeholders.

Keep the name exactly `procedure_clarity`: human feedback is matched to the judge by name.

In [0]:
from mlflow.genai.judges import make_judge

procedure_judge = make_judge(
    name="procedure_clarity",
    # TODO: write your instructions.
    # - The request is in {{ inputs }}
    # - The response is in {{ outputs }}.
    # - State what makes the `procedure` field clear and what makes it excessive.
    # - Then say to answer true only when it is clear and not excessive.
    instructions="""
    The request is in {{ inputs }} and the response is in {{ outputs }}.
    The response is a JSON string with fields "input", "output", and "procedure".
    Focus your evaluation on the "procedure" field.

    The procedure field has CLEAR LOGIC when:
    - The steps follow the order in which the calculation was actually performed.
    - Each step names the specific tool or operation used (e.g. "Called the add tool", "Used the factorial tool").
    - The steps explain how the final result was reached from the input.
    - If the request is non-arithmetic and no tool was needed, the procedure briefly explains why no tool was called.
    - If the request caused an error, the procedure explains what went wrong.

    The procedure field is EXCESSIVE when any of the following are present:
    - The original question or input is restated.
    - Any step or detail is repeated more than once.
    - There is commentary unrelated to the calculation (e.g. general math facts, greetings, or filler).
    - Steps are described that did not actually happen or were not needed.
    - The text exceeds roughly three short sentences for a single-operation request, or five short sentences for a multi-step request.

    Answer true ONLY when the procedure field is clear AND not excessive.
    Answer false when the procedure is unclear, illogical, or contains any excessive content listed above.
""",
    feedback_value_type=bool,  # if your MLflow version rejects this argument, remove it
    model=JUDGE_MODEL,
)

In [0]:
results_procedure = mlflow.genai.evaluate(data=EVAL_DATA, predict_fn=predict_fn, scorers=[procedure_judge])
RUN_PROCEDURE = results_procedure.run_id
outs = outputs_by_question(RUN_PROCEDURE)  # question -> (trace_id, raw reply)
print("run_id:", RUN_PROCEDURE)

2026/10/18 11:37:20 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/11 [Elapsed: 00:00, Remaining: ?]

run_id: 59d9f053d7ff4eb29016fae43f55c8bc


**✏️ Your answer.** Open the run in MLflow and read the `procedure_clarity` rationales.

- Rows judged **not clear / excessive**: TODO
- Does the judge's reasoning look right to you for at least two of them? TODO

# Part 5: Humans vs. the judge

## Step 9: Provide human feedback

Read the model's replies for the six questions below, then give your own verdict using the **same definition** as your judge
(clear logic, not excessive): `True` = the procedure is good, `False` = it is not. Add a one-line reason. Fill in `HUMAN_REVIEW` exactly in this structure (do not rename or remove questions):

In [0]:
QUESTIONS_FOR_HUMAN_REVIEW = [
    "What is 15 plus 27?",
    "What is 5 factorial?",
    "Divide 10 by 0.",
    "What is the factorial of -3?",
    "What is (3 + 4) times 5?",
    "What is the capital of France?",
]

for q in QUESTIONS_FOR_HUMAN_REVIEW:  # read these first
    print(f"Q: {q}\nREPLY: {outs[q][1]}\n{'-' * 70}")

Q: What is 15 plus 27?
REPLY: {"input": "What is 15 plus 27?", "output": 42, "procedure": "Called add with a=15 and b=27, which returned 42."}
----------------------------------------------------------------------
Q: What is 5 factorial?
REPLY: {"input": "What is 5 factorial?", "output": 120, "procedure": "Called factorial with n=5, which returned 120."}
----------------------------------------------------------------------
Q: Divide 10 by 0.
REPLY: {"input": "Divide 10 by 0.", "output": "Error: Division by zero is not allowed", "procedure": "Tried to call divide with a=10 and b=0, but this would result in an error because division by zero is undefined."}
----------------------------------------------------------------------
Q: What is the factorial of -3?
REPLY: {"input": "What is the factorial of -3?", "output": "Error: factorial is not defined for negative numbers", "procedure": "Called factorial with n=-3, which returned an error."}
-------------------------------------------------

In [0]:
REVIEWER = "student@example.com" # put your own email so the feedback has a traceable source

# TODO: replace each (None, "") with (True or False, "one-line reason").
HUMAN_REVIEW = {
    "What is 15 plus 27?":            (True,  "Procedure names the tool, arguments, and result in one concise sentence."),
    "What is 5 factorial?":           (True,  "Procedure names the tool, argument, and result clearly with no extra detail."),
    "Divide 10 by 0.":                (True,  "Procedure explains the tool call and why it failed; slightly verbose but still clear."),
    "What is the factorial of -3?":   (True,  "Procedure states the tool call and that it returned an error, concisely."),
    "What is (3 + 4) times 5?":       (True,  "Procedure lists both steps in order with tools, arguments, and intermediate results."),
    "What is the capital of France?": (True,  "Procedure briefly explains no tool was needed since the question is out of scope."),
}

In [0]:
# PROVIDED: logs your verdicts on the traces. Re-running it adds duplicates, so run it once after filling in.
from mlflow.entities import AssessmentSource, AssessmentSourceType

assert all(v is not None for v, _ in HUMAN_REVIEW.values()), "Fill in every verdict first"

for q, (value, why) in HUMAN_REVIEW.items():
    mlflow.log_feedback(
        trace_id=outs[q][0],
        name="procedure_clarity",  # same name as the judge, so the two can be compared
        value=value,
        rationale=why,
        source=AssessmentSource(source_type=AssessmentSourceType.HUMAN, source_id=REVIEWER),
    )
print("Logged", len(HUMAN_REVIEW), "human verdicts. Open a trace in the UI to see them next to the judge's.")

Logged 6 human verdicts. Open a trace in the UI to see them next to the judge's.


## Step 10: Compare judge and human agreement

Complete the loop below: for each reviewed question, get the judge's verdict on the *same stored reply* and compare it with the human's.

In [0]:
def measure(judge, verbose=True):
    """Agreement between `judge` and HUMAN_REVIEW, on the stored replies."""
    rows = []
    for q, (human_value, _) in HUMAN_REVIEW.items():
        fb = judge(inputs={"question": q}, outputs={"response": outs[q][1]})
        judge_value = as_bool(fb.value)
        rows.append({
            "question": q,
            "human": human_value,
            "judge": judge_value,
            "agree": human_value == judge_value, # TODO: True when the human and the judge gave the same verdict, False otherwise
        })
    df = pd.DataFrame(rows)
    rate = df["agree"].mean()
    if verbose:
        print(f"Agreement: {rate:.0%} ({int(df['agree'].sum())}/{len(df)})")
    return rate, df


rate_v1, df_v1 = measure(procedure_judge)
df_v1

Agreement: 100% (6/6)


,question,human,judge,agree
0,What is 15 plus 27?,True,True,True
1,What is 5 factorial?,True,True,True
2,Divide 10 by 0.,True,True,True
3,What is the factorial of -3?,True,True,True
4,What is (3 + 4) times 5?,True,True,True
5,What is the capital of France?,True,True,True


**✏️ Your answer**

- Agreement rate: TODO
- Disagreements: for each one, say whether the judge was too **strict** or too **lenient**, and quote the part of its rationale that explains it.
  > TODO
- Six labelled rows is a tiny sample. What can and can't you conclude from it?
  > TODO

## Step 11: Alignment (if needed)

**Decide**: if every verdict agreed, say so below and skip to step 12. Otherwise, align the judge.

**Option A: manual alignment (do this one).** Use your disagreements to rewrite the instructions (make the vague parts concrete: lengths, what counts as a step, what to ignore),
create `procedure_judge_v2` (same name!), and re-measure.

In [0]:
procedure_judge_v2 = make_judge(
    name="procedure_clarity",
    instructions="""
    TODO: your improved instructions ({{ inputs }}, {{ outputs }}).
    """,
    feedback_value_type=bool,
    model=JUDGE_MODEL,
)

rate_v2, df_v2 = measure(procedure_judge_v2)
print(f"v1: {rate_v1:.0%}  ->  v2: {rate_v2:.0%}")
df_v2

**✏️ Your answer**

- Decision (skip alignment / align) and why: TODO
- What did you change in the instructions, and which disagreement did each change target? TODO
- Did v2 fix the disagreements without breaking any agreement? Be honest about overfitting to six rows. TODO

**Option B: automatic alignment (optional stretch).** (not given)

Check the docs: https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/alignment/

## Step 12: Tag the traces for the irrelevant questions

Tag the trace for the irrelevant questions (e.g. "What is the capital of France") with:

- Key: question_type
- Value: irrelevant_question

In [0]:
# TODO: find and place the trace IDs for questions containing irrelevant questions in this list
trace_ids = [
    "tr-8cfb1260bf4d3b36cc6c22949868bf35"
]

tagged = 0
for trace_id in trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="question_type", value="irrelevant_question")
    tagged += 1

Tagged 1 traces: ['What is the factorial of 2.5?']


In [0]:
# CHECK (PROVIDED): read the tags back by searching for them.
found = mlflow.search_traces(
    run_id=RUN_PROCEDURE,
    filter_string="tags.question_type = 'irrelevant_question'",
    return_type="list")
print(f"Found {len(found)} traces with the tag (you tagged {tagged}).")

Found 1 traces with the tag (you tagged 1).


**✏️ Your answer**

1. Open the Traces tab and filter by this tag. What do the tagged traces have in common?
   > TODO
2. Was every failure really *verbosity*? If a trace failed for a different reason, what tag would you give it?
   > TODO
3. If you now wanted to fix the agent, what single change would you try first (prompt, tool output, or something else), and which metric would you watch to confirm it worked?
   > TODO

---
### What to hand in
This notebook with all `TODO`s filled, **run top to bottom**, and the `✏️ Your answer` cells completed.